# 循環神經網絡

在上一個單元，我們介紹了文本的豐富語義表示。我們使用的架構捕捉了句子中詞語的綜合意義，但並未考慮詞語的<strong>順序</strong>，因為隨後對嵌入向量進行的聚合操作會從原始文本中去除這些信息。由於這些模型無法表示詞序，它們無法解決更複雜或有歧義的任務，例如文本生成或問答。

為了捕捉文本序列的意義，我們將使用一種稱為<strong>循環神經網絡</strong>（RNN）的神經網絡架構。使用 RNN 時，我們將句子一個詞元一個詞元地傳入網絡，網絡會產生一個<strong>狀態</strong>，然後與下一個詞元一起再次傳入網絡。

![顯示循環神經網絡生成示例的圖片。](../../../../../translated_images/zh-HK/rnn.27f5c29c53d727b5.webp)

給定輸入詞元序列 $X_0,\dots,X_n$，RNN 創建一序列神經網絡塊，並通過反向傳播對該序列進行端到端訓練。每個網絡塊接收一個輸入對 $(X_i,S_i)$，輸出為 $S_{i+1}$。最終狀態 $S_n$ 或輸出 $Y_n$ 會送入線性分類器生成結果。所有網絡塊共享相同權重，並通過一次反向傳播過程端到端訓練。

> 上圖顯示了展開形式的循環神經網絡（左側），以及更緊湊的循環表示法（右側）。重要的是要明白，所有 RNN 單元都共享相同的<strong>可共享權重</strong>。

由於狀態向量 $S_0,\dots,S_n$ 在網絡中傳遞，RNN 能夠學習詞語間的順序依賴關係。例如，當序列中出現詞語 *not* 時，它可以學習在狀態向量中對某些元素進行否定。

每個 RNN 單元內部包含兩個權重矩陣：$W_H$ 和 $W_I$，以及偏差 $b$。在每個 RNN 步驟中，給定輸入 $X_i$ 和輸入狀態 $S_i$，輸出狀態計算為 $S_{i+1} = f(W_H \times S_i + W_I \times X_i + b)$，其中 $f$ 是激活函數（通常是 $\tanh$）。

> 對於文本生成（我們將在下一單元介紹）或機器翻譯等問題，我們希望在每個 RNN 步驟都獲得輸出值。在這種情況下，還有另一個矩陣 $W_O$，輸出計算為 $Y_i = f(W_O \times S_i + b_O)$。

讓我們看看循環神經網絡如何幫助我們分類新聞數據集。

> 在沙盒環境中，我們需要運行以下代碼塊以確保所需庫已安裝，且數據已預取。如果您在本地運行，可以跳過以下代碼塊。


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

當訓練大型模型時，GPU 記憶體分配可能會成為問題。我們亦可能需要嘗試不同的迷你批次大小，使數據能適合我們的 GPU 記憶體，同時訓練速度又足夠快。如果你在自己的 GPU 機器上運行這段代碼，可以嘗試調整迷你批次大小以加快訓練速度。

> <strong>注意</strong>：某些版本的 NVidia 驅動被知悉在訓練模型後不會釋放記憶體。我們在這個筆記本中運行了幾個例子，這可能會導致某些環境下記憶體被耗盡，特別是當你在同一個筆記本中做自己的實驗。如果在開始訓練模型時遇到一些奇怪的錯誤，建議重啟筆記本的核心。


In [3]:
batch_size = 16
embed_size = 64

## 簡單 RNN 分類器

在簡單 RNN 的情況下，每個循環單元都是一個簡單的線性網絡，接受輸入向量和狀態向量，並產生新的狀態向量。在 Keras 中，這可以由 `SimpleRNN` 層表示。

雖然我們可以將獨熱編碼的標記直接傳入 RNN 層，但由於其高維度性，這不是一個好主意。因此，我們將使用嵌入層來降低詞向量的維度，接著是一個 RNN 層，最後是一個 `Dense` 分類器。

> <strong>注意</strong>：在維度不高的情況下，例如使用字元層級的標記化時，直接將獨熱編碼標記傳入 RNN 單元可能是有意義的。


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **注意：** 我們這裡為了簡單起見使用了未經訓練的嵌入層，但為了獲得更好的結果，我們可以使用在前一單元中描述的 Word2Vec 預訓練嵌入層。將此代碼調整為使用預訓練嵌入會是一個很好的練習。

現在讓我們訓練 RNN。一般來說，RNN 較難訓練，因為當 RNN 單元沿序列長度展開時，反向傳播所涉及的層數非常多。因此我們需要選擇較小的學習率，並在較大的數據集上訓練網絡以產生良好結果。這過程可能相當耗時，因此推薦使用 GPU。

為了加快訓練速度，我們將僅在新聞標題上訓練 RNN 模型，省略描述部分。你也可以嘗試用描述來訓練，看看能否成功讓模型學習。


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> <strong>注意</strong> 準確度在這裡可能較低，因為我們僅在新聞標題上進行訓練。


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


現在我們使用了遮罩，便可以用整個標題和描述的數據集來訓練模型。

> <strong>注意</strong>：你有沒有注意到我們一直使用的是在新聞標題上訓練的詞向量化器，而不是整篇文章的內容？這可能會導致一些詞元被忽略，所以最好重新訓練詞向量化器。不過，這可能只有非常小的影響，因此為了簡單起見，我們會保持使用之前預先訓練好的詞向量化器。


## LSTM：長短期記憶

RNN 的主要問題之一是 <strong>梯度消失</strong>。RNN 可能很長，且在反向傳播時可能難以將梯度一路傳回到網絡的第一層。當發生這種情況時，網絡無法學習遙遠標記之間的關係。一種避免此問題的方法是使用 <strong>門控</strong> 引入 <strong>明確的狀態管理</strong>。兩種最常見引入門控的架構是 <strong>長短期記憶</strong>（LSTM）和 <strong>門控循環單元</strong>（GRU）。我們在這裡討論 LSTM。

![展示長短期記憶單元示例的圖片](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM 網絡的組織方式類似於 RNN，但層與層之間傳遞兩個狀態：實際狀態 $c$，以及隱藏向量 $h$。在每個單元中，隱藏向量 $h_{t-1}$ 與輸入 $x_t$ 結合，它們共同透過 <strong>門控</strong> 控制狀態 $c_t$ 與輸出 $h_{t}$ 的變化。每個門控都有 sigmoid 激活（輸出範圍為 $[0,1]$），當與狀態向量相乘時，可視為按位遮罩。LSTM 包含以下門控（見上圖由左至右）：
* <strong>遺忘門</strong>，決定要遺忘向量 $c_{t-1}$ 中哪些組件，哪些組件要保留通過。
* <strong>輸入門</strong>，決定輸入向量和前一隱藏向量中有多少資訊應該被納入狀態向量。
* <strong>輸出門</strong>，以新的狀態向量為依據，決定哪些組件將用來產生新的隱藏向量 $h_t$。

狀態 $c$ 的組件可視作可開關的旗標。例如，當我們在序列中遇到 *Alice* 這個名字時，我們猜測它指的是一位女性，便在狀態中將代表句子有女性名詞的旗標打開。當後續遇到 *and Tom* 時，我們會將代表複數名詞的旗標打開。透過操控狀態，我們就能追蹤句子的語法特性。

> <strong>注意</strong>：這裡有個理解 LSTM 內部原理的好資源：[Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) 由 Christopher Olah 撰寫。

雖然 LSTM 單元的內部結構可能看起來複雜，但 Keras 將實現封裝在 `LSTM` 層中，因此在上述範例中，我們要做的只是替換遞歸層：


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> <strong>注意</strong> 訓練 LSTM 同樣相當緩慢，並且在訓練初期你可能不會見到準確率有太大提升。你可能需要繼續訓練一段時間才能達到良好的準確率。


## 雙向及多層 RNN

在我們目前的範例中，循環網絡都是從序列的開始運行到結束。這對我們來說很自然，因為它遵循了我們閱讀或聆聽語音的方向。然而，對於需要隨機存取輸入序列的場景，雙向運算的循環計算會更合理。允許雙向計算的 RNN 稱為 <strong>雙向</strong> RNN，可以透過用特殊的 `Bidirectonal` 層包裝循環層來建立。

> <strong>注意</strong>：`Bidirectional` 層會在其中複製出兩個該層的副本，並將其中一個副本的 `go_backwards` 屬性設為 `True`，使其沿序列的相反方向運行。

不論是單向或雙向的循環網絡，都會捕捉序列中的模式並將它們儲存到狀態向量或作為輸出返回。就像卷積網絡一樣，我們可以在第一層之後再構建另一層循環層，以捕捉由第一層提取的較低層模式所組成的較高層模式。這就引出了<strong>多層 RNN</strong>的概念，它由兩層或多層循環網絡組成，前一層的輸出會作為後一層的輸入。

![顯示多層長短期記憶 RNN 的圖片](../../../../../translated_images/zh-HK/multi-layer-lstm.dd975e29bb2a59fe.webp)

*圖片來自 Fernando López 的[這篇精彩文章](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3)。*

Keras 讓建構這些網絡變得簡單，因為你只需要向模型加入更多循環層。除了最後一層外，對所有層都需要指定 `return_sequences=True` 參數，因為我們需要該層返回所有中間狀態，而不只是循環計算的最終狀態。

現在讓我們為分類問題建構一個兩層的雙向 LSTM。

> <strong>注意</strong> 這段程式碼執行時間仍然相當長，但它給我們帶來了目前為止最高的準確率。所以或許值得等待並看結果。


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## 用於其他任務的 RNN

到目前為止，我們一直專注於使用 RNN 來對文本序列進行分類。但它們可以處理更多任務，例如文本生成和機器翻譯——我們會在下一單元考慮這些任務。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
